In [1]:
import pandas as pd
from stairs_semantic_mapping.infrastructure import TEIEmbeddingModel, NormalizingEmbeddingModel
from stairs_semantic_mapping.utils.settings import settings

In [2]:
def prepare_data_for_loading(hierarchy_df_lst: list[tuple[pd.DataFrame, int]],
                             emb_model: NormalizingEmbeddingModel) -> pd.DataFrame:
    codes_lst = []
    names_lst = []
    code_hierarchical_work_types_lst = []
    names_for_embeddings = []

    for df, level in hierarchy_df_lst:
        for i in range(len(df)):
            codes_lst.append(df.loc[i, 'code'])
            names_lst.append(df.loc[i, 'name'])
            code_hierarchical_work_types_lst.append(level)
            names_for_embeddings.append('passage: ' + df.loc[i, 'code'] + ' || ' + df.loc[i, 'name'])

    embeddings = [[str(emb) for emb in emb_model.embed(['passage: ' + x])[0]] for x in names_for_embeddings]

    df_all_hierarchies = pd.DataFrame()
    df_all_hierarchies['code'] = codes_lst
    df_all_hierarchies['name'] = names_lst
    df_all_hierarchies['code_hierarchical_work_types'] = code_hierarchical_work_types_lst
    df_all_hierarchies['embeddings'] = embeddings

    return df_all_hierarchies

# Load data

In [3]:
# Load dataframes
df_bwd = pd.read_csv('bwd_names.csv')
df_occ = pd.read_csv('occ_names.csv')
df_oce = pd.read_csv('oce_names.csv')

hierarchy_df_lst = [(df_bwd, 3), (df_oce, 2), (df_occ, 1)]
emb_model_base = TEIEmbeddingModel(host=settings.embedding_host_hierarchies)
emb_model = NormalizingEmbeddingModel(emb_model_base)

In [4]:
all_df = prepare_data_for_loading(hierarchy_df_lst, emb_model)
all_df.to_csv('all_hierarchies_base.csv', index=False)